# Simple – The Price Is Right 🛍️💰


In this notebook, I fine-tune **DistilBERT** (a lightweight transformer model) to estimate product prices based on their descriptions.

### What we'll do

1. **Load the data** – Start with Amazon product listings data
2. **Prepare for fine-tuning** – Split data into train/validation/test sets
3. **Set a baseline** – Our "lazy" model always guesses the **average price**
4. **Fine-tune the model** – Train **DistilBERT** with a regression head on our data
5. **Evaluate performance** – Compare predictions using **Mean Absolute Error (MAE)**
6. **Interactive demo** – a **Gradio UI** to test the fine-tuned model

By the end, you'll see how fine-tuning an open-source model performs on real-world price prediction tasks.

In [ ]:
import random
import numpy as np
import torch
import gradio as gr
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tqdm.auto import tqdm

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Model configuration
MODEL_NAME = 'distilbert-base-uncased'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## Step 1: Load and Clean Data

In [ ]:
# Load Amazon Appliances product listings from HuggingFace

dataset = load_dataset(
    'McAuley-Lab/Amazon-Reviews-2023',
    'raw_meta_Appliances',
    split='full',
    trust_remote_code=True
)

print(f'Total products data: {len(dataset):,}')

In [ ]:
# Clean data: product must have a price between $1-$1000, and an elaborate description

def clean(item):
    try:
        price = float(item.get("price"))
    except (TypeError, ValueError):
        return None

    if not 1 <= price <= 1000:
        return None

    text = " ".join([
        item.get("title", "").strip(),
        *item.get("features", []),
        *item.get("description", [])
    ]).strip()

    if len(text) < 50:
        return None

    return {
        "text": text[:512],  # Limit to 512 chars for efficiency
        "price": price
    }

items = [clean(d) for d in dataset]
items = [i for i in items if i]
print(f'Clean products data: {len(items):,}')

# Sample for training (use more if you have compute resources)
random.shuffle(items)
items = items[:2000]  # Adjust based on your compute budget
print(f'Using {len(items)} samples for this demo')

## Step 2: Split Data into Train/Val/Test Sets

In [ ]:
# Split: 70% train, 15% validation, 15% test

train_size = int(0.7 * len(items))
val_size = int(0.15 * len(items))

train_items = items[:train_size]
val_items = items[train_size:train_size + val_size]
test_items = items[train_size + val_size:]

print(f'Train set: {len(train_items)}')
print(f'Validation set: {len(val_items)}')
print(f'Test set: {len(test_items)}')

# Calculate statistics for normalization
train_prices = [i['price'] for i in train_items]
mean_price = np.mean(train_prices)
std_price = np.std(train_prices)

print(f'\nTrain set statistics:')
print(f'Mean price: ${mean_price:.2f}')
print(f'Std price: ${std_price:.2f}')
print(f'Min: ${min(train_prices):.2f}, Max: ${max(train_prices):.2f}')

## Step 3: Baseline Model (Always Predict Mean)

In [ ]:
# Baseline: always predict the mean price

test_actuals = [i['price'] for i in test_items]
baseline_preds = [mean_price] * len(test_items)

baseline_mae = mean_absolute_error(test_actuals, baseline_preds)
baseline_rmse = np.sqrt(mean_squared_error(test_actuals, baseline_preds))

print(f'Baseline (always predict mean):')
print(f'  MAE: ${baseline_mae:.2f}')
print(f'  RMSE: ${baseline_rmse:.2f}')

## Step 4: Prepare Data for Fine-Tuning

In [ ]:
# Initialize tokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Create HuggingFace datasets
train_dataset = Dataset.from_dict({
    'text': [i['text'] for i in train_items],
    'label': [i['price'] for i in train_items]
})

val_dataset = Dataset.from_dict({
    'text': [i['text'] for i in val_items],
    'label': [i['price'] for i in val_items]
})

test_dataset = Dataset.from_dict({
    'text': [i['text'] for i in test_items],
    'label': [i['price'] for i in test_items]
})

# Tokenize function
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print('Data tokenized and ready for training!')

## Step 5: Fine-Tune the Model

In [ ]:
# Load pre-trained model with regression head (num_labels=1 for regression)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type="regression"
)

model.to(DEVICE)
print(f'Model loaded on {DEVICE}')

In [ ]:
# Define training arguments

training_args = TrainingArguments(
    output_dir='./price_predictor_checkpoints',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,
    report_to='none',
    seed=42
)

# Define metrics for evaluation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.flatten()
    
    mae = mean_absolute_error(labels, predictions)
    rmse = np.sqrt(mean_squared_error(labels, predictions))
    r2 = r2_score(labels, predictions)
    
    return {
        'mae': mae,
        'rmse': rmse,
        'r2': r2
    }

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('Trainer initialized. Starting training...')

In [ ]:
# Train the model

trainer.train()
print('Training complete!')

## Step 6: Evaluate on Test Set

In [ ]:
# Evaluate on test set

test_results = trainer.predict(test_dataset)
test_predictions = test_results.predictions.flatten()

test_mae = mean_absolute_error(test_actuals, test_predictions)
test_rmse = np.sqrt(mean_squared_error(test_actuals, test_predictions))
test_r2 = r2_score(test_actuals, test_predictions)

print('=' * 60)
print('TEST SET RESULTS')
print('=' * 60)
print(f'Baseline (always predict mean):')
print(f'  MAE: ${baseline_mae:.2f}')
print(f'  RMSE: ${baseline_rmse:.2f}')
print()
print(f'Fine-tuned DistilBERT:')
print(f'  MAE: ${test_mae:.2f}')
print(f'  RMSE: ${test_rmse:.2f}')
print(f'  R² Score: {test_r2:.4f}')
print()
improvement = ((baseline_mae - test_mae) / baseline_mae) * 100
print(f'Improvement over baseline: {improvement:.1f}%')
print('=' * 60)

In [ ]:
# Show sample predictions

print('\nSample predictions on test set:')
print(f'{"Actual":>10}  {"Predicted":>10}  {"Error":>10}  {"Description"}')
print('-' * 80)

for i in range(min(15, len(test_items))):
    actual = test_actuals[i]
    predicted = test_predictions[i]
    error = abs(actual - predicted)
    desc = test_items[i]['text'][:50] + '...'
    print(f'${actual:>9.2f}  ${predicted:>9.2f}  ${error:>9.2f}  {desc}')

## Step 7: Save the Fine-Tuned Model

In [ ]:
# Save model and tokenizer

model.save_pretrained('./fine_tuned_price_predictor')
tokenizer.save_pretrained('./fine_tuned_price_predictor')

print('Model and tokenizer saved to ./fine_tuned_price_predictor')

## Step 8: Interactive Demo with Gradio

In [ ]:
# Create prediction function for Gradio

def predict_price(description):
    if not description.strip():
        return "Please enter a product description."
    
    # Tokenize input
    inputs = tokenizer(
        description,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=128
    ).to('cpu')
    
    # Make prediction
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_price = outputs.logits.item()
    
    return f"Predicted Price: ${predicted_price:.2f}"

# Example product descriptions
examples = [
    ["Stainless steel electric kettle with auto shut-off and 1.7L capacity"],
    ["Wireless noise cancelling over-ear Bluetooth headphones with 30 hour battery life"],
    ["Compact air fryer with digital touchscreen and 5 quart capacity"],
    ["USB-C fast charging cable 6ft braided for iPhone and Android"],
    ["Robot vacuum cleaner with smart mapping and WiFi control"],
    ["Programmable coffee maker with thermal carafe and 12-cup capacity"]
]

# Create Gradio interface
demo = gr.Interface(
    fn=predict_price,
    inputs=gr.Textbox(
        lines=5,
        label="Product Description",
        placeholder="Enter an Amazon product description..."
    ),
    outputs=gr.Textbox(label="Price Prediction"),
    examples=examples,
    title="Fine-Tuned Price Predictor",
    description="""
This model was fine-tuned on Amazon product data using DistilBERT.
Enter a product description to get a price prediction.
"""
)

demo.launch()